Tạo dữ liệu và kiểm tra bài toán xếp đặt
Đây là bộ 4 task/2 host trong đề cương Topic 16 của bạn. Nhu cầu và dung lượng phải sử dụng cùng đơn vị cho từng loại tài nguyên.     nt409-topic16-proposal

In [1]:
import numpy as np
import pandas as pd
from itertools import product
from IPython.display import display

tasks = pd.DataFrame(
    {
        "cpu": [2, 3, 1, 4],
        "ram": [4, 2, 3, 5],
    },
    index=["T1", "T2", "T3", "T4"],
)

hosts = pd.DataFrame(
    {
        "cpu": [5, 6],
        "ram": [6, 8],
    },
    index=["A", "B"],
)

display(tasks)
display(hosts)

resources = ["cpu", "ram"]

demand = tasks[resources].to_numpy()
capacity = hosts[resources].to_numpy()

n_tasks = len(tasks)
n_hosts = len(hosts)

,cpu,ram
T1,2,4
T2,3,2
T3,1,3
T4,4,5


,cpu,ram
A,5,6
B,6,8


Quy ước 0 = host A, 1 = host B. Các vị trí lần lượt tương ứng T1, T2, T3, T4.
Cell thứ hai — viết hàm kiểm tra:

In [2]:
def evaluate_placement(placement):
    if len(placement) != n_tasks:
        raise ValueError("Cần gán host cho đầy đủ các task.")

    usage = np.zeros_like(capacity)

    for task_id, host_id in enumerate(placement):
        if host_id not in range(n_hosts):
            raise ValueError(f"Host không hợp lệ: {host_id}")

        usage[host_id] += demand[task_id]

    valid_per_host = (usage <= capacity).all(axis=1)

    report = pd.DataFrame(
        {
            "cpu_used": usage[:, 0],
            "cpu_capacity": capacity[:, 0],
            "ram_used": usage[:, 1],
            "ram_capacity": capacity[:, 1],
            "valid": valid_per_host,
        },
        index=hosts.index,
    )

    return bool(valid_per_host.all()), report

Cell thứ ba — kiểm tra một phương án hợp lệ và một phương án quá tải:

In [3]:
for placement in [(0, 0, 1, 1), (0, 0, 0, 1)]:
    feasible, report = evaluate_placement(placement)

    print(f"Placement: {placement}")
    print(f"Hợp lệ: {feasible}")
    display(report)

Placement: (0, 0, 1, 1)
Hợp lệ: True


,cpu_used,cpu_capacity,ram_used,ram_capacity,valid
A,5,5,6,6,True
B,5,6,8,8,True


Placement: (0, 0, 0, 1)
Hợp lệ: False


,cpu_used,cpu_capacity,ram_used,ram_capacity,valid
A,6,5,9,6,False
B,4,6,5,8,True


Duyệt toàn bộ phương án

In [4]:
feasible_placements = []

for placement in product(range(n_hosts), repeat=n_tasks):
    feasible, _ = evaluate_placement(placement)

    if feasible:
        feasible_placements.append(placement)

print("Tổng số phương án:", n_hosts ** n_tasks)
print("Số phương án hợp lệ:", len(feasible_placements))
print("Các phương án hợp lệ:", feasible_placements)

Tổng số phương án: 16
Số phương án hợp lệ: 1
Các phương án hợp lệ: [(0, 0, 1, 1)]
